# Tutorial 23: Building a Custom Pool Backend

LAILA's pool base class (`_LAILA_IDENTIFIABLE_POOL`) defines a small set of hooks. Override them, register the new class with a nickname, and `memorize` / `remember` / `forget` against it work just like every other backend.

You will:

- Subclass `_LAILA_IDENTIFIABLE_POOL` for an in-memory dict-backed pool
- Implement the six sync hooks
- Add a true async `_write_async` override
- Round-trip a numpy array, a torch tensor, and a dict through the new backend

**Prerequisites:** `pip install laila-core`.

In [ ]:
import asyncio
from typing import Optional, Any, Iterable

from pydantic import PrivateAttr

import laila
from laila.pool.schema.base import _LAILA_IDENTIFIABLE_POOL

## Minimum viable pool

The defaults on `_LAILA_IDENTIFIABLE_POOL` already implement an in-memory dict-backed pool, so the bare minimum override is... nothing. But that's not very instructive. We'll override every hook explicitly so it's clear what each does, and back the store with a private dict so we can inspect it externally.

In [ ]:
class DictPool(_LAILA_IDENTIFIABLE_POOL):
    """A toy pool backed by a private dict."""

    _store: dict = PrivateAttr(default_factory=dict)

    def _read(self, key: str) -> Optional[Any]:
        with self.atomic():
            return self._store.get(key)

    def _write(self, key: str, value: Any) -> None:
        with self.atomic():
            self._store[key] = value

    def _delete(self, key: str) -> None:
        with self.atomic():
            self._store.pop(key, None)

    def _exists(self, key: str) -> bool:
        with self.atomic():
            return key in self._store

    def _keys(self, as_generator: bool = False) -> Iterable[str]:
        with self.atomic():
            return list(self._store.keys())

    def _empty(self) -> None:
        with self.atomic():
            self._store.clear()

    async def _write_async(self, key: str, value: Any) -> None:
        await asyncio.sleep(0)  # yield to the event loop
        self._write(key, value)


print("DictPool subclassed _LAILA_IDENTIFIABLE_POOL")

## Register and round-trip

In [ ]:
import numpy as np
import torch

pool = DictPool(nickname="dict_pool")
laila.memory.extend(pool, pool_nickname="dict_pool")

entries = [
    laila.constant(data=np.arange(8), nickname="cp_array"),
    laila.constant(data=torch.randn(2, 3), nickname="cp_tensor"),
    laila.constant(data={"label": "demo", "score": 0.95}, nickname="cp_dict"),
]
for e in entries:
    laila.memorize(e, pool_nickname="dict_pool").wait()

print("keys in pool:", len(list(pool._keys())))

In [ ]:
for nick in ["cp_array", "cp_tensor", "cp_dict"]:
    r = laila.remember(nickname=nick, pool_nickname="dict_pool", persist=False).wait()
    if isinstance(r, list):
        r = r[0]
    print(f"  {nick}: type={type(r.data).__name__}")

## Inspecting your backing store

Because we used a private dict, the pool's payload is observable from the outside if you want to debug. (A real backend would wrap a database client or network handle here.)

In [ ]:
for k, v in list(pool._store.items())[:1]:
    print("  key prefix:", k[:60], "...")
    print("  value type:", type(v).__name__)

## Async paths

Default `_read_async` / `_write_async` / `_delete_async` just call the sync hook on the calling thread. That's correct but blocks the loop for the duration of the call. We overrode `_write_async` above to demonstrate the pattern — replace `asyncio.sleep(0)` with a real `await` against an async client (`aioboto3`, `asyncpg`, `motor`, etc.) for true non-blocking I/O.

## Packaging as a separate distribution

A custom pool can ship in its own pip package — there is no plug-in registry. Users `pip install my-pool-pkg`, `from my_pool import MyPool`, and pass it to `laila.memory.extend(...)`. The `transformations` field is inherited automatically, so callers can wrap any custom backend in encryption or compression without touching your class.

## Summary

- Six sync hooks (`_read`, `_write`, `_delete`, `_exists`, `_keys`, `_empty`) cover the local API.
- Four async hooks (`_read_async`, `_write_async`, `_delete_async`, `_exists_async`) default to the sync versions; override them for genuine non-blocking I/O.
- Custom pools work with the routing, manifest, and proxy machinery out of the box.